## load_silver_zillow
Conforms the three long Bronze feeds (`zillow_zhvi`/`zori`/`inventory`) into `silver.fact_zillow_metro_monthly`, full-outer-joined on `(region_id, period_date)`. Joins `dim_geo` via **`zillow_region_id`** (not `cbsa_code`).

**Transforms:** the Bronze `value` is already typed DOUBLE, so each feed's value is rounded to BIGINT (`round(value,0).cast(long)`) into `typical_home_value` / `typical_rent` / `inventory_active`; `period_date` is already month-end → `date_key`. Filtered to `region_type='msa'` (the Zillow US national row is not a metro). Region with no CBSA crosswalk match → `silver.quarantine` (`unmatched_geography`). MERGE on `(geo_key, date_key)`.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
STEP_SEQUENCE = 1
SOURCE_SYSTEM = "zillow"
SOURCE_TABLE  = f"{BRONZE}.zillow_*"   # 3 feeds; descriptive only (source_table audit label)
TARGET_TABLE  = f"{SILVER}.fact_zillow_metro_monthly"
QUARANTINE    = f"{SILVER}.quarantine"
DIM_GEO       = f"{SILVER}.dim_geo"
VALUE_COLS    = ["typical_home_value", "typical_rent", "inventory_active"]
# feed table -> fact value column (Bronze value is DOUBLE; round to whole BIGINT).
FEEDS = {
    "zillow_zhvi":      "typical_home_value",
    "zillow_zori":      "typical_rent",
    "zillow_inventory": "inventory_active",
}

In [ ]:
# Open the pipeline_step_log row (RUNNING).
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "silver",
    target_table    = TARGET_TABLE,
)
print(f"load_silver_zillow: step_log_id={step.step_log_id}")

In [ ]:
# Outer-join the 3 metro-only feeds on (region_id, period_date), derive date_key, resolve
# geo_key via dim_geo.zillow_region_id. No string casts (Bronze value is already DOUBLE), so
# the only quarantine reason here is an unmatched region.
try:
    def feed(tbl, alias):
        return (spark.table(f"{BRONZE}.{tbl}").where("region_type = 'msa'")
                .select("region_id", "period_date",
                        F.round(F.col("value"), 0).cast("long").alias(alias)))

    items = list(FEEDS.items())
    joined = feed(*items[0])
    for tbl, alias in items[1:]:
        joined = joined.join(feed(tbl, alias), ["region_id", "period_date"], "fullouter")
    rows_read = joined.count()

    date_key = (F.year("period_date") * 10000 + F.month("period_date") * 100
                + F.dayofmonth("period_date")).cast("int")
    geo = spark.table(DIM_GEO).select("geo_key", F.col("zillow_region_id").alias("region_id"))
    staged = (joined.withColumn("date_key", date_key)
                    .join(geo, "region_id", "left"))
    step.rows_read = rows_read
    print(f"load_silver_zillow: read {rows_read:,} (region, period) rows across 3 feeds")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Split on geo match, quarantine unmatched regions, MERGE on (geo_key, date_key), audit.
transform_started = datetime.now(timezone.utc)
rows_rejected = rows_inserted = rows_updated = 0
try:
    good = staged.where(F.col("geo_key").isNotNull())
    bad  = staged.where(F.col("geo_key").isNull())
    rows_rejected = bad.count()

    spark.sql(f"DELETE FROM {QUARANTINE} WHERE source_system = '{SOURCE_SYSTEM}'")
    if rows_rejected > 0:
        bad.select(
            F.expr("uuid()").alias("quarantine_id"),
            F.lit(SOURCE_SYSTEM).alias("source_system"),
            F.lit(None).cast("string").alias("source_file_path"),
            F.concat_ws("|", F.col("region_id"), F.col("period_date").cast("string")).alias("natural_key"),
            F.to_json(F.struct("region_id", "period_date", *VALUE_COLS)).alias("raw_payload"),
            F.lit("unmatched_geography").alias("quarantine_reason"),
            F.current_timestamp().alias("quarantined_ts"),
        ).write.format("delta").mode("append").saveAsTable(QUARANTINE)

    fact_cols = ["geo_key", "date_key"] + VALUE_COLS
    good.select(*[F.col(c) for c in fact_cols],
                F.current_timestamp().alias("inserted_ts"),
                F.current_timestamp().alias("updated_ts")).createOrReplaceTempView("zillow_fact_staging")

    set_clause = ", ".join(f"t.{c}=s.{c}" for c in VALUE_COLS) + ", t.updated_ts=s.updated_ts"
    cols_csv   = ", ".join(fact_cols + ["inserted_ts", "updated_ts"])
    vals_csv   = ", ".join(f"s.{c}" for c in fact_cols + ["inserted_ts", "updated_ts"])
    metrics = spark.sql(f"""
        MERGE INTO {TARGET_TABLE} t USING zillow_fact_staging s
        ON t.geo_key = s.geo_key AND t.date_key = s.date_key
        WHEN MATCHED THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED THEN INSERT ({cols_csv}) VALUES ({vals_csv})
    """).first().asDict()
    rows_inserted = metrics.get("num_inserted_rows") or 0
    rows_updated  = metrics.get("num_updated_rows") or 0

    step.rows_written = rows_inserted
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=rows_inserted, rows_inserted=rows_inserted, rows_updated=rows_updated,
        rows_rejected=rows_rejected, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"load_silver_zillow: inserted={rows_inserted:,} updated={rows_updated:,} "
          f"quarantined={rows_rejected:,} (read={step.rows_read:,})")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_rejected=rows_rejected, error_message=f"{type(e).__name__}: {e}",
        ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise